# Quadratic Discriminant Analysis model

 Access the documentation to learn more: https://scikit-learn.org/stable/modules/lda_qda.html

In [1]:
# importing relevant libraries
# in terminal: uv sync
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import recall_score, f1_score, roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE

# Ensure project root is importable regardless of notebook working directory
project_root = Path.cwd().resolve()
while not (project_root / 'app').exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.services.custom_qda import CustomQDA

## 1. Define the CustomQDA class

In [2]:
# CustomQDA is imported from app.services.custom_qda so saved models are portable
print("CustomQDA class imported from app.services.custom_qda")

CustomQDA class imported from app.services.custom_qda


## 2. Load raw data and define variables 

In [3]:
df_abt = pd.read_csv("../../data/processed/aml_clean.csv")

# Convert date and time as done previously 
df_abt['date'] = pd.to_datetime(df_abt['date']).dt.date
df_abt['time'] = pd.to_datetime(df_abt['time'], format='%H:%M:%S').dt.time

# Drop rows where 'is_laundering' is NaN
df_abt_cleaned = df_abt.dropna(subset=['is_laundering'])

key_vars = ['date', 'time', 's_bank_name', 's_bank_id', 's_entity_id',
       's_entity_name', 's_entity_type','s_latitude',
       's_longitude', 'r_bank_name', 'r_bank_id', 'r_entity_id',
       'r_entity_name', 'r_entity_type', 'r_latitude', 'r_longitude']

cat_vars = ['from_bank', 'account', 'to_bank', 'account.1',
       'receiving_currency', 'payment_currency', 'currency_mismatch', 'payment_format','s_country','r_country',
       's_bank_type','r_bank_type']

num_vars = ["amount_received" ,"amount_paid"]

target = "is_laundering"

In [4]:
# Stratified split to create training and test sets
# Note: This is for demonstrating the full pipeline, in a real Flask app,
# we would only use the training part to generate the models once.
subset_proportion = 0.1 # Using 10% for demonstration/memory
if subset_proportion < 1.0:
    df_abt_subset, _ = train_test_split(df_abt_cleaned, train_size=subset_proportion, random_state=42, stratify=df_abt_cleaned['is_laundering'])
else:
    df_abt_subset = df_abt_cleaned.copy()

train_df, test_df = train_test_split(df_abt_subset, test_size=0.2, random_state=42, stratify=df_abt_subset['is_laundering'])

print("Data loaded and split into train/test.")


Data loaded and split into train/test.


## 3. Preprocessing pipeline construction and training data preparation

In [5]:
# The preprocessing pipeline was extracted from PyCaret. To replicate exactly without PyCaret setup,
# we would build an sklearn.Pipeline with similar transformers. For simplicity and exact replication
# of the *saved* PyCaret preprocessing, we load the saved pipeline.

loaded_preprocessing_pipeline = joblib.load('../../analysis/features/pycaret_preprocessing_pipeline.pkl')

# Prepare training and test data by dropping ignored features and target
X_train_raw = train_df.drop(columns=key_vars + [target])
y_train = train_df[target]
X_test_raw = test_df.drop(columns=key_vars + [target])
y_test = test_df[target]

# Transform raw training data using the loaded preprocessing pipeline
X_train_preprocessed = loaded_preprocessing_pipeline.transform(X_train_raw)
X_test_preprocessed = loaded_preprocessing_pipeline.transform(X_test_raw)

print("Data preprocessed using the saved pipeline.")


Data preprocessed using the saved pipeline.


## 4. Apply SMOTE to preprocessed training data 


In [6]:
smote_instance = SMOTE(k_neighbors=3, random_state=42, sampling_strategy=0.2)
X_train_smoted, y_train_smoted = smote_instance.fit_resample(X_train_preprocessed, y_train)

print("SMOTE applied to training data.")

SMOTE applied to training data.


## 5. Train the model

In [7]:
custom_qda_model = CustomQDA(reg_param=0.14) # the tuned reg_param
custom_qda_model.fit(X_train_smoted, y_train_smoted)

print("Custom QDA Model trained successfully.")


Custom QDA Model trained successfully.


In [8]:
joblib.dump(custom_qda_model, '../../analysis/models/custom_qda_model_for_flask.pkl')


['../../analysis/models/custom_qda_model_for_flask.pkl']

## 6. Cross-validation 

In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_raw, y_train), start=1):
    X_fold_train_raw = X_train_raw.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val_raw = X_train_raw.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Preprocess within each fold to avoid leakage
    X_fold_train_preprocessed = loaded_preprocessing_pipeline.transform(X_fold_train_raw)
    X_fold_val_preprocessed = loaded_preprocessing_pipeline.transform(X_fold_val_raw)

    # Apply SMOTE only on the fold's training split
    smote_fold = SMOTE(k_neighbors=3, random_state=42, sampling_strategy=0.2)
    X_fold_train_smoted, y_fold_train_smoted = smote_fold.fit_resample(X_fold_train_preprocessed, y_fold_train)

    fold_model = CustomQDA(reg_param=0.14)
    fold_model.fit(X_fold_train_smoted, y_fold_train_smoted)

    y_fold_pred = fold_model.predict(X_fold_val_preprocessed)
    proba_matrix = fold_model.predict_proba(X_fold_val_preprocessed)
    class_one_idx = list(fold_model.classes).index(1)
    y_fold_proba = proba_matrix[:, class_one_idx]

    cv_rows.append({
        'fold': fold,
        'recall': recall_score(y_fold_val, y_fold_pred),
        'f1': f1_score(y_fold_val, y_fold_pred),
        'auc': roc_auc_score(y_fold_val, y_fold_proba),
        'accuracy': accuracy_score(y_fold_val, y_fold_pred)
    })

cv_results_df = pd.DataFrame(cv_rows)
print("5-Fold Stratified CV Results (on training set):")
print(cv_results_df)
print("\nMean CV Metrics:")
print(cv_results_df[['recall', 'f1', 'auc', 'accuracy']].mean())

5-Fold Stratified CV Results (on training set):
   fold    recall        f1       auc  accuracy
0     1  0.867470  0.014748  0.904289  0.881607
1     2  0.879518  0.015027  0.929047  0.882223
2     3  0.902439  0.014818  0.938721  0.878898
3     4  0.927711  0.015796  0.943963  0.881914
4     5  0.831325  0.014497  0.910275  0.884547

Mean CV Metrics:
recall      0.881693
f1          0.014977
auc         0.925259
accuracy    0.881838
dtype: float64


## 7. Testing

In [10]:
# Make predictions on the preprocessed test set
y_pred_custom_qda = custom_qda_model.predict(X_test_preprocessed)
y_proba_custom_qda = custom_qda_model.predict_proba(X_test_preprocessed)[:, 1]  # Probability of class 1

# Evaluate the custom model
recall = recall_score(y_test, y_pred_custom_qda)
f1 = f1_score(y_test, y_pred_custom_qda)
auc = roc_auc_score(y_test, y_proba_custom_qda)
accuracy = accuracy_score(y_test, y_pred_custom_qda)

print("Custom QDA Model Performance on Test Set:")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

# Display a few predictions for illustration
print("\nSample predictions (first rows from test set):")
n_samples_to_show = min(10, len(y_test))
for i in range(n_samples_to_show):
    print(f"Actual: {y_test.iloc[i]}, Predicted: {y_pred_custom_qda[i]}, Prob(Laundering): {y_proba_custom_qda[i]:.4f}")

Custom QDA Model Performance on Test Set:
Recall: 0.8558
F1-Score: 0.0145
AUC: 0.9202
Accuracy: 0.8812

Sample predictions (first rows from test set):
Actual: 0, Predicted: 0, Prob(Laundering): 0.2884
Actual: 0, Predicted: 1, Prob(Laundering): 0.9914
Actual: 0, Predicted: 0, Prob(Laundering): 0.0467
Actual: 0, Predicted: 0, Prob(Laundering): 0.0198
Actual: 0, Predicted: 0, Prob(Laundering): 0.0365
Actual: 0, Predicted: 0, Prob(Laundering): 0.0197
Actual: 0, Predicted: 0, Prob(Laundering): 0.0015
Actual: 0, Predicted: 0, Prob(Laundering): 0.0365
Actual: 0, Predicted: 0, Prob(Laundering): 0.1616
Actual: 0, Predicted: 0, Prob(Laundering): 0.0271
